# afriseg 02: train one arm on BraTS 2021, then test it on BraTS-Africa

**Before running:**
1. Settings: **Accelerator = GPU T4 x2 or GPU P100**, **Internet on**.
2. Add Input:
   - **Your Work, Notebooks:** the saved output of `afriseg 01: prepare data and calibrate`
   - **Your Datasets:** your BraTS-Africa dataset (`datasite1`)
3. Set `ARM` in the first code cell.
4. **Save Version, then Save & Run All (Commit).** One arm takes roughly 6 to 10 hours.

**Order:** run `A1_generic` and `A2_15t` first. They are the primary comparison.
Then run `A0_none`, `A2u_15t_prior` and `A3_both`.

**If a run hits the time limit,** add that version's output as an extra input and commit again
with the same `ARM`. Training resumes from the last finished epoch.


In [ ]:
# 0. Choose the experiment arm (one per commit)
ARM = 'A1_generic'   # A0_none | A1_generic | A2_15t | A2u_15t_prior | A3_both

EPOCHS = 150         # identical for every arm; do not change between arms
ITERS = 250          # patches per epoch (batch 2)
MAX_HOURS = 10.0     # stop training here, leaving time for evaluation within Kaggle's 12 h

CAL = '/kaggle/working/afriseg/configs/aug15t_calibrated.json'
ARMS = {
    'A0_none':        ('none',        None),
    'A1_generic':     ('generic',     None),
    'A2_15t':         ('15t',         CAL),
    'A2u_15t_prior':  ('15t',         None),
    'A3_both':        ('15t+generic', CAL),
}
AUG, AUG_CONFIG = ARMS[ARM]
print(ARM, '->', AUG, AUG_CONFIG or '(default ranges)')

In [ ]:
# 1. Code and GPU
!rm -rf /kaggle/working/afriseg && git clone -q https://github.com/AJmoe/afriseg.git /kaggle/working/afriseg
!pip install -q -e /kaggle/working/afriseg
!cd /kaggle/working/afriseg && git log --oneline -1
import os, sys, glob, json, shutil, subprocess
os.environ['PYTHONPATH'] = '/kaggle/working/afriseg/src'
sys.path.insert(0, '/kaggle/working/afriseg/src')
import torch
assert torch.cuda.is_available(), 'No GPU: set Accelerator to GPU in Settings'
print(torch.cuda.get_device_name(0))

In [ ]:
# 2. Find the prepared data
def find(pattern):
    hits = sorted(glob.glob('/kaggle/input/**/' + pattern, recursive=True))
    assert hits, f'Not found: {pattern}. Is the afriseg 01 output attached?'
    return hits[0]

M21 = find('npz/brats2021_manifest.csv')
S21 = os.path.join(os.path.dirname(os.path.dirname(M21)), 'splits', 'brats2021.json')
assert os.path.exists(S21), S21
MAF = find('africa_manifest.csv')
for name, p in [('BraTS 2021 manifest', M21), ('BraTS 2021 split', S21), ('BraTS-Africa manifest', MAF)]:
    print(f'{name:22s} {p}')
os.makedirs('/kaggle/working/runs', exist_ok=True)
os.makedirs('/kaggle/working/results', exist_ok=True)
OUT = f'/kaggle/working/runs/{ARM}'

In [ ]:
# 3. Resume from an earlier, unfinished version of this arm if one is attached
RESUME = []
prev = [p for p in glob.glob(f'/kaggle/input/**/runs/{ARM}/last.pt', recursive=True)]
if prev and not os.path.exists(OUT + '/DONE'):
    src = os.path.dirname(sorted(prev, key=os.path.getmtime)[-1])
    if os.path.exists(os.path.join(src, 'DONE')):
        print('Attached run is already finished:', src)
    else:
        print('Resuming from', src)
    shutil.copytree(src, OUT, dirs_exist_ok=True)
    RESUME = ['--resume']
else:
    print('Fresh run')

In [ ]:
# 4. Train (source only: BraTS 2021 train split, best epoch picked on BraTS 2021 val split)
cmd = ['python', '-m', 'afriseg.train', '--manifest', M21, '--split', S21,
       '--subset', 'train', '--val-subset', 'val', '--aug', AUG,
       '--epochs', str(EPOCHS), '--iters', str(ITERS), '--workers', '4',
       '--max-hours', str(MAX_HOURS), '--out', OUT] + RESUME
if AUG_CONFIG:
    cmd += ['--aug-config', AUG_CONFIG]
if not os.path.exists(OUT + '/DONE'):
    print(' '.join(cmd), flush=True)
    subprocess.run(cmd, check=True)
print('finished' if os.path.exists(OUT + '/DONE') else 'NOT finished: commit again with this output attached')

In [ ]:
# 5. Evaluate the selected checkpoint on all 95 BraTS-Africa cases and the BraTS 2021 test split
if os.path.exists(OUT + '/DONE'):
    subprocess.run(['python', '-m', 'afriseg.evaluate', '--ckpt', OUT + '/best.pt', '--manifest', MAF,
                    '--arm', ARM, '--out', f'/kaggle/working/results/{ARM}_africa.csv'], check=True)
    subprocess.run(['python', '-m', 'afriseg.evaluate', '--ckpt', OUT + '/best.pt', '--manifest', M21,
                    '--split', S21, '--subset', 'test', '--arm', ARM,
                    '--out', f'/kaggle/working/results/{ARM}_brats_test.csv'], check=True)
    import pandas as pd
    for name in ('africa', 'brats_test'):
        df = pd.read_csv(f'/kaggle/working/results/{ARM}_{name}.csv')
        d = df[['dice_WT', 'dice_TC', 'dice_ET']]
        print(f'{ARM} on {name}: n={len(df)}  mean Dice WT/TC/ET =',
              d.mean().round(3).tolist(), ' overall =', round(d.mean(axis=1).mean(), 3))

## Afterwards
Download `results/<ARM>_africa.csv` and `results/<ARM>_brats_test.csv` from the Output tab, or paste
the last printed lines into the chat. Once both A1 and A2 are done, the primary test is:

```
python -m afriseg.stats --a results/A1_generic_africa.csv --b results/A2_15t_africa.csv
```
